# 02 - Statik kare seçimi
Hareket kayıtlarında düşük hareketli bölgeleri aday gösterir. Eşikler veri seti kare hızı ve validasyon sonuçlarıyla gerekçelendirilmelidir.

In [ ]:
import sys, zipfile
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    artifacts_root = Path('/content/drive/MyDrive/PhysioLoad_AI_Project')
    code_root = Path('/content/physioload-ai')
    if not (code_root / 'physioload_env.py').exists():
        code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(artifacts_root / 'code' / 'physioload-source.zip') as package: package.extractall(code_root)
else:
    artifacts_root = None
    here = Path.cwd().resolve()
    code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))
from physioload_env import setup_environment
env = setup_environment(root=code_root, artifacts_root=artifacts_root)
print(env.summary())

In [ ]:
import cv2
import numpy as np

def motion_score(previous_rgb, current_rgb):
    prev = cv2.cvtColor(previous_rgb, cv2.COLOR_RGB2GRAY)
    curr = cv2.cvtColor(current_rgb, cv2.COLOR_RGB2GRAY)
    flow = cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    return float(np.median(np.linalg.norm(flow, axis=2)))

def static_windows(scores, window=5, quantile=0.20):
    scores = np.asarray(scores, float)
    if len(scores) < window:
        return []
    rolling = np.convolve(scores, np.ones(window) / window, mode='valid')
    threshold = float(np.quantile(rolling, quantile))
    return [(int(i), float(value)) for i, value in enumerate(rolling) if value <= threshold]

print('Fonksiyonlar hazır. Video/kare dizisini inventory kayıtlarıyla besleyin.')

In [ ]:
# Sentetik kontrol: düşük hareketli orta bölge seçilmelidir.
demo_scores = [2.1, 1.8, 0.2, 0.15, 0.12, 0.18, 1.9, 2.2]
static_windows(demo_scores, window=3, quantile=0.35)